# UTS Machine Learning 2026 – Task 1: Estimasi Waktu Pengiriman (`durasi_jam`)

**Platform:** Python di VS Code (`.venv`) &nbsp;|&nbsp; **Modul Utama:** `kode/regresi.py` & `kode/config.py`

**Tujuan:** Memprediksi median (`p50`) dan batas atas 90% (`p90`) waktu pengiriman (`durasi_jam`) untuk paket uji **September–Oktober 2025** dengan **hanya menggunakan fitur yang tersedia saat paket diambil kurir (pickup)**, tanpa kebocoran data masa depan.

| Bagian | Cakupan Ketentuan UTS | Deskripsi |
|---|---|---|
| 0 | R3 | Persiapan lingkungan, self-healing `sys.path`, versi library eksplisit, dan `SEED = 42` |
| 1 | R1a | Load data latih (Jan–Agu 2025) & uji (Sep–Okt 2025) + ringkasan kamus data |
| 2 | R1a | Audit kebocoran (`jumlah_scan`), kesalahan satuan 60x (`durasi_jam > 1000`), sentinel `jarak_km`, MNAR `berat_kg`, drift `f30`/`f35`, & kategori tak terlihat (`K13`) |
| 3 | R1b | Validasi berbasis waktu (*expanding-window next-month Time CV*) vs Random K-Fold CV |
| 4 | R1c | Feature engineering saat pickup, OOF `TargetEncoder`, baseline + 3 model kuantil lanjutan (*HGBR, LightGBM, XGBoost*) |
| 5 | R1d | Evaluasi Pinball Loss (`q=0.5`, `q=0.9`), cakupan empiris P90, dan kalibrasi Split Conformal (CQR) |
| 6 | R1e | Rincian error per `layanan`, `pulau_tujuan`, `mitra_kurir` (termasuk `K13`), serta interpretabilitas SHAP & PDP |
| 7 | R1f | Ekspor & verifikasi `prediksi_regresi.csv` (`id_paket,p50,p90`, `p90 >= p50`) |
| 8 | R3 | Ringkasan metodologi, keterbatasan, dan reproduksi satu perintah (`python kode/run_all.py`) |

## 0. Persiapan, Versi Library, & Konfigurasi

**Tujuan:** Menyiapkan `PROJECT_ROOT` secara otomatis, mengimpor konfigurasi terpusat (`kode.config`) dan pipeline modular (`kode.regresi`), menetapkan random seed `SEED = 42`, serta mencatat seluruh versi library classical ML yang digunakan.

In [1]:
from pathlib import Path
import sys

def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "kode" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Direktori root proyek yang berisi kode/config.py tidak ditemukan.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from IPython.display import display
from sklearn.inspection import PartialDependenceDisplay

from kode.config import (
    DEFAULT_REGRESI_CSV,
    OUTPUT_DIR,
    SEED,
    format_library_versions,
    get_data_dir,
    get_library_versions,
    get_smoke_config,
    set_global_seed,
)
from kode.regresi import (
    _predict_with_bundle,
    apply_conformal_calibration,
    audit_and_clean_regresi,
    calibrate_conformal_intervals,
    compute_group_breakdowns,
    compute_interpretability,
    evaluate_time_vs_random_cv,
    export_regresi_csv,
    load_regresi_datasets,
    train_and_tune_quantile_models,
)

# ====== KONFIGURASI EKSEKUSI (ubah SMOKE_TEST=True untuk uji cepat) ======
SMOKE_TEST = False
SAMPLE_N = None
DATA_DIR = get_data_dir(must_exist=True)
ARTIFACTS_DIR = OUTPUT_DIR / "regresi_artifacts"
OUTPUT_CSV = DEFAULT_REGRESI_CSV
# ========================================================================

set_global_seed(SEED)
sns.set_theme(style="whitegrid")
lib_versions = get_library_versions()
smoke_cfg = get_smoke_config(smoke_test=SMOKE_TEST, sample_n=SAMPLE_N, seed=SEED)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"OUTPUT_CSV   : {OUTPUT_CSV}")
print(f"SEED         : {SEED}")
print(f"SMOKE_TEST   : {SMOKE_TEST} (sample_n={SAMPLE_N})")
print(f"Versi Library: {format_library_versions(lib_versions)}")

PROJECT_ROOT : C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI
DATA_DIR     : C:\Users\rafai\Downloads\data_uts_ml
OUTPUT_CSV   : C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\prediksi_regresi.csv
SEED         : 42
SMOKE_TEST   : False (sample_n=None)
Versi Library: python=3.14.5, numpy=2.5.5, pandas=3.1.1, scipy=1.18.1, scikit-learn=1.9.1, lightgbm=4.7.0, xgboost=3.4.1, shap=0.54.1, imbalanced-learn=0.14.2, matplotlib=3.11.2, seaborn=0.13.2, joblib=1.6.0, pytest=9.2.0


## 1. Memuat Dataset Regresi (`regresi_train.csv.gz` & `regresi_test.csv.gz`)

**Tujuan:** Membaca data latih dan uji dengan presisi `float64` (tanpa downcasting prematur) dan memeriksa rentang tanggal `tgl_pickup` serta kelengkapan kolom.

In [2]:
train_raw, test_raw = load_regresi_datasets(DATA_DIR, smoke_cfg=smoke_cfg)

print(f"Train shape: {train_raw.shape[0]:,} baris x {train_raw.shape[1]} kolom")
print(f"Test  shape: {test_raw.shape[0]:,} baris x {test_raw.shape[1]} kolom")
print(f"Rentang tgl_pickup train: {train_raw['tgl_pickup'].min()} s.d. {train_raw['tgl_pickup'].max()}")
print(f"Rentang tgl_pickup test : {test_raw['tgl_pickup'].min()} s.d. {test_raw['tgl_pickup'].max()}")
display(train_raw.head(3))

Train shape: 250,000 baris x 77 kolom
Test  shape: 60,000 baris x 76 kolom
Rentang tgl_pickup train: 2025-01-01 s.d. 2025-08-31
Rentang tgl_pickup test : 2025-09-01 s.d. 2025-10-31


,id_paket,tgl_pickup,jam_pickup,layanan,mitra_kurir,hub_asal,kota_asal,pulau_tujuan,kota_tujuan,jarak_km,...,f42,f43,f44,f45,f46,f47,f48,f49,f50,durasi_jam
0,P24198861,2025-03-13,10,ekspres,K07,JKT,KA01,Jawa,KT099,1077.6,...,0.683,0.644,3.957,1.0,3.0,-0.207,0.721,0.661,3.760,40.4
1,P44313480,2025-06-24,9,reguler,K05,SBY,KA09,Jawa,KT101,829.3,...,-0.521,0.011,0.429,0.0,2.0,0.890,0.065,0.232,1.866,78.7
2,P18332724,2025-05-07,11,kargo,K03,JKT,KA02,Jawa,KT066,758.6,...,-1.004,0.506,0.795,0.0,2.0,-2.769,-1.031,0.777,2.809,101.3


## 2. Audit Kualitas Data & Kebocoran Fitur (R1a)

**Tujuan:** Mengidentifikasi dan mengukur dampak masalah kualitas data serta fitur pasca-pickup:
1. **Kebocoran pasca-pickup (`jumlah_scan`)**: tercatat setelah proses pengiriman berlangsung (`train` berkorelasi kuat dengan `durasi_jam`, sedangkan `test` bernilai kecil `<= 5`). Jika dipakai, prediksi uji akan runtuh secara artifisial. Fitur ini **wajib dibuang** dari seluruh fitur pemodelan.
2. **Kesalahan satuan 60x pada target (`durasi_jam > 1000`)**: nilai ekstrem ribuan jam seluruhnya merupakan menit/satuan 60x; dibuktikan melalui uji kelipatan 60 (`(y * 10) % 60 == 0`), celah distribusi `(700, 1000]`, serta korelasi terhadap `jarak_km` sebelum vs sesudah perbaikan `/ 60.0` (khusus data latih).
3. **Sentinel `jarak_km` (`0.0` & `9999.0`)**: dipulihkan secara hierarkis hanya dari median rute valid di fold latih (`(kota_asal, kota_tujuan) -> (hub_asal, kota_tujuan) -> (kota_asal, pulau_tujuan) -> pulau_tujuan -> global`) + indikator `jarak_km_is_anomalous`.
4. **Missingness MNAR `berat_kg` pada `kargo`**: diimputasi berbasis median `(layanan, volume_bin)` khusus fold latih + indikator `berat_kg_is_missing`.
5. **Pergeseran distribusi (`f30`, `f35`, `antrean_hub_asal`) & kategori uji baru (`mitra_kurir = 'K13'`, `kota_tujuan` 405 level)**.

In [3]:
train_clean, test_clean, audit_summary = audit_and_clean_regresi(
    train_raw, test_raw, seed=SEED, smoke_cfg=smoke_cfg
)

print("Kolom kebocoran yang dibuang:", audit_summary["dropped_leakage_columns"])
print("\n--- Bukti Kebocoran jumlah_scan ---")
display(pd.DataFrame([audit_summary["jumlah_scan"]]).drop(columns=["impact"], errors="ignore"))
if audit_summary["jumlah_scan"].get("impact"):
    print("Dampak jumlah_scan pada validasi OOT:")
    display(pd.DataFrame([audit_summary["jumlah_scan"]["impact"]]))

print("\n--- Bukti Kesalahan Satuan durasi_jam > 1000 ---")
display(pd.DataFrame([audit_summary["durasi_jam_unit_errors"]]).drop(columns=["impact"], errors="ignore"))
if audit_summary["durasi_jam_unit_errors"].get("impact"):
    print("Dampak pembersihan durasi_jam > 1000 pada validasi OOT:")
    display(pd.DataFrame([audit_summary["durasi_jam_unit_errors"]["impact"]]))

print("\n--- Sentinel jarak_km, MNAR berat_kg, Drift & Profil Kategori ---")
display(pd.DataFrame([audit_summary["jarak_km_sentinels"]]))
display(pd.DataFrame(audit_summary["temporal_drift"]).T)
print("Profil kategori:", audit_summary["categorical_profile"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(np.log1p(train_raw["durasi_jam"]), bins=50, alpha=0.6, label="Raw durasi_jam")
axes[0].hist(np.log1p(train_clean["durasi_jam"]), bins=50, alpha=0.6, label="Cleaned durasi_jam")
axes[0].set_title("Distribusi log1p(durasi_jam): Sebelum vs Sesudah Perbaikan 60x")
axes[0].set_xlabel("log1p(durasi_jam)")
axes[0].legend()

axes[1].hist(train_raw["jumlah_scan"], bins=30, alpha=0.6, density=True, label="Train jumlah_scan")
axes[1].hist(test_raw["jumlah_scan"], bins=15, alpha=0.7, density=True, label="Test jumlah_scan")
axes[1].set_title("Shift Kebocoran jumlah_scan (Train vs Test)")
axes[1].set_xlabel("jumlah_scan (dibuang dari fitur model)")
axes[1].legend()
plt.tight_layout()
plt.show()

Kolom kebocoran yang dibuang: ['jumlah_scan']

--- Bukti Kebocoran jumlah_scan ---


,train_mean,train_median,train_max,test_mean,test_median,test_max,shift_z,corr_clean_durasi,action
0,9.651072,8.0,97.0,1.502,1.0,7.0,-1.44872,0.851796,dropped_post_pickup_leakage


Dampak jumlah_scan pada validasi OOT:


,oot_pinball_p50_with_leak,oot_pinball_p50_without_leak,oot_mae_with_leak,oot_mae_without_leak,oot_mae_under_test_scan_shift,mean_pred_collapse_under_scan_shift
0,6.344148,7.956435,12.688296,15.912869,25.144092,-16.110029



--- Bukti Kesalahan Satuan durasi_jam > 1000 ---


,threshold_hours,count_gt_1000,count_gap_700_1000,frac_integer_gt_1000,frac_div60_tenths_gt_1000,raw_max,raw_mean,clean_max,clean_mean,corr_jarak_before,corr_jarak_after,action
0,1000.0,370,0,1.0,1.0,25452.0,77.838195,693.1,71.265925,0.106937,0.529868,divided_gt_1000_by_60_on_train_only


... (3 outputs left out)

,train_zero_count,train_9999_count,test_zero_count,test_9999_count,test_recovery_breakdown,action
0,703,456,177,130,"{'total_anomalous': 307, 'level_1_kota_pair': ...",train_only_hierarchical_route_median_plus_flag


,train_mean,train_std,test_mean,test_std,shift_z,corr_clean_durasi
f30,0.496592,1.036897,1.091638,1.005288,0.573872,-0.001792
f35,1.005590,0.087789,1.028548,0.110041,0.261510,0.097941
antrean_hub_asal,3328.676856,7028.812791,4618.875900,8298.146822,0.183559,0.148525


Profil kategori: {'kota_tujuan_train_nunique': 405, 'kota_tujuan_test_nunique': 405, 'unseen_kota_tujuan': [], 'unseen_kota_tujuan_count': 0, 'mitra_kurir_train_levels': ['K01', 'K02', 'K03', 'K04', 'K05', 'K06', 'K07', 'K08', 'K09', 'K10', 'K11', 'K12'], 'unseen_mitra_kurir': ['K13'], 'unseen_mitra_kurir_test_rows': 2342, 'unseen_mitra_kurir_test_pct': 0.03903333333333334}


C:\Users\rafai\AppData\Local\Temp\ipykernel_28736\906129145.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Validasi Berbasis Waktu vs Random Cross-Validation (R1b)

**Tujuan:** Merancang skema validasi *expanding-window next-month Time-Based CV* (melatih dari bulan awal hingga bulan `t-1` untuk memprediksi bulan `t`, sehingga `train_max < val_min`) dan membandingkannya secara langsung dengan *Random K-Fold CV* pada fitur dan model yang identik untuk mengukur bias optimisme (*optimism gap*).

In [4]:
cv_summary = evaluate_time_vs_random_cv(train_clean, seed=SEED, smoke_cfg=smoke_cfg)

print("Jendela Fold Time-Based CV (Expanding Window):")
display(pd.DataFrame(cv_summary["time_cv"]["folds"]))

df_time = pd.DataFrame(cv_summary["time_cv"]["models"]).assign(scheme="Time-Based CV")
df_rand = pd.DataFrame(cv_summary["random_cv"]["models"]).assign(scheme="Random K-Fold CV")
df_cv_comp = pd.concat([df_time, df_rand], ignore_index=True)
display(df_cv_comp)

print("Optimism Gap (Time CV - Random CV):")
display(pd.DataFrame(cv_summary["optimism_gap"]))

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=df_cv_comp, x="model", y="pinball_avg", hue="scheme", ax=ax)
ax.set_title("Perbandingan Pinball Loss Rata-Rata: Time-Based CV vs Random K-Fold CV")
plt.tight_layout()
plt.show()

2026-11-20 20:23:29 | INFO | uts_ml.regresi | [Step 3/6] Evaluating Time-Based CV (3 folds) vs Random CV (3 folds) on 40000 rows...
2026-11-20 20:25:49 | INFO | uts_ml.regresi |   -> Time-Based CV completed.
2026-11-20 20:27:13 | INFO | uts_ml.regresi |   -> Random CV completed.


Jendela Fold Time-Based CV (Expanding Window):


,fold,train_n,val_n,train_min,train_max,val_min,val_max,val_label
0,1,23910,5110,2025-01-01,2025-05-31,2025-06-01,2025-06-30,2025-06
1,2,29020,5514,2025-01-01,2025-06-30,2025-07-01,2025-07-31,2025-07
2,3,34534,5466,2025-01-01,2025-07-31,2025-08-01,2025-08-31,2025-08


,model,pinball_p50,pinball_p90,pinball_avg,mae,rmse,p90_coverage,scheme
0,Baseline_RidgeLogQuantile,7.529381,4.631962,6.080671,15.058762,25.827714,0.878875,Time-Based CV
1,HistGradientBoosting,6.906000,4.542095,5.724047,13.812000,24.432614,0.867752,Time-Based CV
2,LightGBM,6.871186,4.484021,5.677604,13.742373,24.236609,0.868731,Time-Based CV
3,Baseline_RidgeLogQuantile,7.439833,4.630673,6.035253,14.879667,25.076600,0.900100,Random K-Fold CV
4,HistGradientBoosting,6.775312,4.412990,5.594151,13.550625,23.819523,0.880075,Random K-Fold CV
5,LightGBM,6.733136,4.356379,5.544757,13.466271,23.714028,0.881775,Random K-Fold CV


Optimism Gap (Time CV - Random CV):


,model,pinball_avg_gap,mae_gap,rmse_gap,p90_cov_diff
0,Baseline_RidgeLogQuantile,0.045418,0.179095,0.751114,-0.021225
1,HistGradientBoosting,0.129896,0.261376,0.613091,-0.012323
2,LightGBM,0.132846,0.276102,0.522582,-0.013044


C:\Users\rafai\AppData\Local\Temp\ipykernel_28736\2449144976.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Feature Engineering, Kategori Berkardinalitas Tinggi, & Pemodelan Kuantil (R1c)

**Tujuan:**
- Seluruh transformasi stateful (`RouteJarakImputer`, `BeratMNARImputer`, frekuensi rute, dan OOF `TargetEncoder` pada `kota_tujuan`, `kota_asal`, `mitra_kurir`) dipasang **ketat di dalam setiap fold latih** (*zero leakage*).
- `kota_tujuan` (405 level) dipertahankan sebagai fitur numerik `_te` + `_freq` agar tidak melanggar batas `max_bins=255` pada `HistGradientBoostingRegressor`.
- Kategori baru pada data uji (`mitra_kurir = 'K13'`, atau `kota_tujuan` baru) ditangani aman oleh fallback global prior (`TargetEncoder`), frekuensi `0.0`, `OrdinalEncoder(unknown_value=-1)`, dan masking `pd.Categorical`.
- Membandingkan 2 baseline (`DummyRegressor`, `RidgeLogQuantile`) dan 3 model lanjutan (`HistGradientBoostingRegressor`, `LGBMRegressor`, `XGBRegressor`) dengan tuning hanya di dalam Time-Based CV pada skala `log1p(durasi_jam)`.

In [5]:
model_results, best_model_bundle = train_and_tune_quantile_models(
    train_clean, seed=SEED, smoke_cfg=smoke_cfg
)

print("Strategi Encoding:", model_results["encoding_strategy"])
print("Model Terpilih   :", model_results["selected_model"])
print("Hyperparameter   :", model_results["selected_params"])

comp_df = pd.DataFrame(model_results["comparison_table"])
display(comp_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
comp_melt = comp_df.melt(
    id_vars=["model"],
    value_vars=["pinball_p50", "pinball_p90"],
    var_name="kuantil",
    value_name="pinball_loss",
)
sns.barplot(data=comp_melt, x="model", y="pinball_loss", hue="kuantil", ax=axes[0])
axes[0].set_title("Perbandingan Pinball Loss (Time-Based CV)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=comp_df, x="model", y="mae", ax=axes[1], color="#4dabf7")
axes[1].set_title("MAE Prediksi Median P50 (jam)")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

2026-11-20 20:27:17 | INFO | uts_ml.regresi | [Step 4/6] Tuning 5 model families across 3 Time-CV folds (50000 tuning rows, 250000 full rows)...
2026-11-20 20:27:27 | INFO | uts_ml.regresi |   -> [Tuned] Baseline_Dummy             | pinball_avg=11.2146 (p50=13.3610, p90=9.0681) | P90_cov=0.900
2026-11-20 20:28:08 | INFO | uts_ml.regresi |   -> [Tuned] Baseline_RidgeLogQuantile  | pinball_avg=6.0509 (p50=7.4887, p90=4.6132) | P90_cov=0.882
2026-11-20 20:31:07 | INFO | uts_ml.regresi |   -> [Tuned] HistGradientBoosting       | pinball_avg=5.6411 (p50=6.8209, p90=4.4613) | P90_cov=0.875
2026-11-20 20:32:15 | INFO | uts_ml.regresi |   -> [Tuned] LightGBM                   | pinball_avg=5.5955 (p50=6.7721, p90=4.4188) | P90_cov=0.868
2026-11-20 20:34:11 | INFO | uts_ml.regresi |   -> [Tuned] XGBoost                    | pinball_avg=5.8689 (p50=7.0206, p90=4.7172) | P90_cov=0.878
2026-11-20 20:34:12 | INFO | uts_ml.regresi |   -> Fitting winning model (LightGBM) on full OOT fold (215832 trai

Strategi Encoding: OOF TargetEncoder(cv=KFold, smooth='auto') + train-only freq for kota_tujuan(405)/kota_asal/mitra_kurir; OrdinalEncoder for HGBR; masked pd.Categorical for LGBM/XGB; OHE+StandardScaler for linear baseline; unseen K13/kota_tujuan fallback to global prior & freq=0
Model Terpilih   : LightGBM
Hyperparameter   : {'n_estimators': 250, 'learning_rate': 0.06, 'num_leaves': 31, 'min_child_samples': 40, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0}


,model,is_baseline,params,pinball_p50,pinball_p90,pinball_avg,mae,rmse,p90_coverage,mean_width
0,LightGBM,False,"{'n_estimators': 250, 'learning_rate': 0.06, '...",6.772119,4.418848,5.595484,13.544238,23.638226,0.868222,22.463405
1,HistGradientBoosting,False,"{'max_iter': 220, 'learning_rate': 0.06, 'max_...",6.820906,4.461349,5.641128,13.641811,23.773775,0.875229,23.567456
2,XGBoost,False,"{'n_estimators': 280, 'learning_rate': 0.05, '...",7.020595,4.717222,5.868908,14.041190,24.335065,0.877855,24.518246
3,Baseline_RidgeLogQuantile,True,{'alpha': 10.0},7.488676,4.613218,6.050947,14.977353,25.441647,0.881642,27.239242
4,Baseline_Dummy,True,{},13.361001,9.068135,11.214568,26.722002,41.133300,0.899516,56.346663


C:\Users\rafai\AppData\Local\Temp\ipykernel_28736\1497575739.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Ketidakpastian Kuantil (`p50`, `p90`) & Kalibrasi Conformal (R1d)

**Tujuan:** Mengukur Pinball Loss (`q=0.5` dan `q=0.9`), cakupan empiris P90 (`mean(y <= p90)`), serta melakukan kalibrasi *Split Conformal* (membandingkan mode aditif vs *log-space CQR* pada paruh kronologis out-of-time) untuk memastikan cakupan empiris mendekati target `0.90` dan menjamin ketidaksetaraan `p90 >= p50`.

In [6]:
oot_val_idx = best_model_bundle["oot_val_idx"]
q_hat, oot_p90_cal, calib_summary = calibrate_conformal_intervals(
    y_cal=best_model_bundle["oot_y"],
    p50_cal=best_model_bundle["oot_p50_raw"],
    p90_cal_raw=best_model_bundle["oot_p90_raw"],
    tgl_cal=train_clean.iloc[oot_val_idx]["tgl_pickup"].to_numpy(),
    alpha=0.90,
)

print(f"Ruang Kalibrasi Terpilih : {calib_summary['selected_space']}")
print(f"Conformal q_hat Produksi : {q_hat:.5f}")
display(pd.DataFrame({
    "sebelum_kalibrasi": calib_summary["raw_metrics"],
    "sesudah_kalibrasi": calib_summary["calibrated_metrics"],
}).T)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
cov_before = calib_summary["raw_metrics"]["p90_coverage"]
cov_after = calib_summary["calibrated_metrics"]["p90_coverage"]
ax.bar(["Sebelum Kalibrasi", "Sesudah Conformal"], [cov_before, cov_after], color=["#5c7cfa", "#20c997"])
ax.axhline(0.90, color="crimson", linestyle="--", label="Target 0.90")
ax.set_ylim(0.0, 1.05)
ax.set_title("Cakupan Empiris P90 Out-of-Time (Sebelum vs Sesudah Conformal)")
ax.legend()
plt.tight_layout()
plt.show()

Ruang Kalibrasi Terpilih : log_space
Conformal q_hat Produksi : 0.01298


,pinball_p50,pinball_p90,pinball_avg,mae,rmse,p90_coverage,mean_width
sebelum_kalibrasi,6.277018,4.002900,5.139959,12.554035,22.287012,0.892297,22.481216
sesudah_kalibrasi,6.277018,3.996563,5.136790,12.554035,22.287012,0.900433,23.669207


C:\Users\rafai\AppData\Local\Temp\ipykernel_28736\3235445684.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Rincian Error per Segmen & Interpretabilitas SHAP / Partial Dependence (R1e)

**Tujuan:** Menganalisis error (`pinball_p50`, `pinball_p90`, `mae`, `rmse`, `p90_hit_rate`) berdasarkan `layanan`, `pulau_tujuan`, dan `mitra_kurir` (termasuk perilaku pada kurir baru `K13` di data uji), serta menjelaskan penggerak utama model `p50` menggunakan **SHAP (`TreeExplainer`)** dan **Partial Dependence Plot (PDP)**.

In [7]:
full_prep = best_model_bundle["full_preprocessor"]
test_views = full_prep.transform(test_clean)
test_p50_raw, test_p90_raw = _predict_with_bundle(best_model_bundle["full_bundle"], test_views)
test_p50, test_p90 = apply_conformal_calibration(
    test_p50_raw,
    test_p90_raw,
    q_hat=q_hat,
    mode=str(calib_summary["selected_space"]),
)

seen_mitra = set(train_clean["mitra_kurir"].astype(str).unique())
breakdowns = compute_group_breakdowns(
    val_df_meta=train_clean.iloc[oot_val_idx],
    y_val=best_model_bundle["oot_y"],
    p50_val=best_model_bundle["oot_p50_raw"],
    p90_val=oot_p90_cal,
    test_df=test_clean,
    p50_test=test_p50,
    p90_test=test_p90,
    seen_mitra=seen_mitra,
)

print("--- Rincian Error OOT per Layanan ---")
display(pd.DataFrame(breakdowns["by_layanan"]))
print("--- Rincian Error OOT per Pulau Tujuan ---")
display(pd.DataFrame(breakdowns["by_pulau_tujuan"]))
print("--- Ringkasan Prediksi Uji per Mitra Kurir (termasuk K13 unseen) ---")
display(pd.DataFrame(breakdowns["test_mitra_kurir_summary"]))

interp_summary, saved_artifacts = compute_interpretability(
    best_model_bundle=best_model_bundle,
    seed=SEED,
    smoke_cfg=smoke_cfg,
    cv_summary=cv_summary,
    model_results=model_results,
    calib_summary=calib_summary,
    oot_p90_cal=oot_p90_cal,
    breakdowns=breakdowns,
    artifacts_dir=ARTIFACTS_DIR,
)

print("\nTop 10 Fitur Penggerak Utama (SHAP):")
display(pd.DataFrame(interp_summary["top_drivers"]).head(10))
print("Ringkasan Partial Dependence (dalam jam):")
display(pd.DataFrame(interp_summary["pdp_summary"]))
print("Artefak grafik tersimpan di:", ARTIFACTS_DIR)
for k, p in saved_artifacts.items():
    print(f"  - {k}: {p}")

--- Rincian Error OOT per Layanan ---


,group,n,mean_actual,median_actual,mean_p50,mean_p90,pinball_p50,pinball_p90,mae,rmse,p90_hit_rate,mean_interval_width
0,ekspres,5070,44.618738,39.1,41.303158,57.932299,4.229101,3.340512,8.458203,19.858912,0.901972,16.629141
1,hemat,11349,84.853503,73.5,80.716637,112.102841,8.379705,4.890649,16.759411,27.151943,0.899374,31.386204
2,kargo,3414,91.774634,79.2,86.664364,114.373625,7.496669,4.608707,14.993339,27.465157,0.898067,27.709261
3,reguler,14335,65.903872,58.3,62.865607,83.953892,5.562359,3.662941,11.124718,19.594048,0.900384,21.088285


--- Rincian Error OOT per Pulau Tujuan ---


,group,n,mean_actual,median_actual,mean_p50,mean_p90,pinball_p50,pinball_p90,mae,rmse,p90_hit_rate,mean_interval_width
0,Bali-Nusa Tenggara,1257,76.249483,66.2,71.798070,102.409509,7.615921,4.532973,15.231842,25.638731,0.916468,30.611438
1,Jawa,22917,62.192425,56.4,58.922728,77.894362,5.079517,3.508866,10.159034,19.271042,0.898983,18.971634
2,Kalimantan,1981,98.032610,92.1,93.081132,133.958375,10.627322,6.000069,21.254645,30.829292,0.893488,40.877242
3,Maluku,679,121.441826,114.2,119.863355,170.630276,12.128452,6.221514,24.256905,31.807770,0.930781,50.766921
4,Papua,603,179.054561,163.4,167.353787,239.397300,20.433825,11.081136,40.867651,60.161359,0.885572,72.043514
5,Sulawesi,2181,97.918478,88.7,93.239299,131.818372,10.195811,5.572559,20.391622,29.913794,0.895002,38.579073
6,Sumatera,4550,72.081077,64.2,68.544661,95.166305,7.043024,4.313385,14.086048,22.986918,0.903516,26.621644


--- Ringkasan Prediksi Uji per Mitra Kurir (termasuk K13 unseen) ---


,group,n,mean_p50,median_p50,mean_p90,median_p90,mean_interval_width,is_unseen_in_train
0,K01,7943,64.789086,57.187067,88.316172,77.692385,23.527086,False
1,K02,6944,72.969509,66.046588,98.535461,86.129019,25.565952,False
2,K03,6341,72.032065,65.502201,97.384079,85.419076,25.352014,False
3,K04,5728,66.557182,59.425494,89.773718,79.308397,23.216536,False
4,K05,5302,71.135902,63.632590,96.901732,84.086677,25.765830,False
5,K06,4630,70.327196,63.642931,95.651824,83.860396,25.324628,False
6,K07,4660,84.965684,78.329731,113.342377,101.229826,28.376693,False
7,K08,4003,59.135776,51.820792,81.254244,71.928330,22.118469,False
8,K09,3484,65.984892,59.016933,89.878467,79.580316,23.893575,False
9,K10,3433,70.870092,63.592099,96.241237,83.923746,25.371146,False



Top 10 Fitur Penggerak Utama (SHAP):


,feature,mean_abs_shap
0,layanan,0.181564
1,jarak_km,0.111016
2,kota_tujuan_te,0.081009
3,f17,0.054113
4,cod,0.034153
5,mitra_kurir_te,0.031553
6,pulau_tujuan,0.029687
7,jarak_per_pengalaman,0.029167
8,f04,0.026523
9,antrean_hub_asal,0.023589


Ringkasan Partial Dependence (dalam jam):


,feature,grid_min,grid_max,pdp_min_hours,pdp_max_hours,pdp_range_hours
0,jarak_km,100.687000,2848.336000,53.280128,78.961009,25.680881
1,kota_tujuan_te,3.886490,4.635534,56.038889,83.259822,27.220934
2,f17,0.000000,1.258540,65.582574,81.083862,15.501288
3,mitra_kurir_te,4.021185,4.232008,62.566354,71.035953,8.469599


Artefak grafik tersimpan di: C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\output\regresi_artifacts
  - cv_and_model_comparison: C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\output\regresi_artifacts\cv_and_model_comparison.png
  - conformal_coverage_diagnostics: C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\output\regresi_artifacts\conformal_coverage_diagnostics.png
  - error_breakdown_groups: C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\output\regresi_artifacts\error_breakdown_groups.png
  - shap_summary_p50: C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\output\regresi_artifacts\shap_summary_p50.png
  - pdp_top_features: C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\output\regresi_artifacts\pdp_top_features.png


## 7. Ekspor & Verifikasi `prediksi_regresi.csv` (R1f)

**Tujuan:** Menyimpan hasil prediksi akhir ke `prediksi_regresi.csv` dengan header persis `id_paket,p50,p90`, jumlah baris dan urutan ID persis sama dengan `regresi_test.csv.gz`, tanpa NaN/Inf, `p50 > 0`, dan `p90 >= p50`.

In [8]:
written_csv = export_regresi_csv(
    ids=test_clean["id_paket"],
    p50=test_p50,
    p90=test_p90,
    output_csv=OUTPUT_CSV,
)

assert Path(written_csv).name == "prediksi_regresi.csv"
df_sub = pd.read_csv(written_csv, encoding="utf-8")
assert list(df_sub.columns) == ["id_paket", "p50", "p90"], f"Header salah: {list(df_sub.columns)}"
assert len(df_sub) == len(test_clean), f"Jumlah baris tidak cocok: {len(df_sub)} vs {len(test_clean)}"
assert (df_sub["id_paket"].astype(str).to_numpy() == test_clean["id_paket"].astype(str).to_numpy()).all()
assert np.isfinite(df_sub["p50"]).all() and np.isfinite(df_sub["p90"]).all()
assert (df_sub["p50"] > 0).all() and (df_sub["p90"] >= df_sub["p50"]).all()

print(f"File submission berhasil ditulis: {written_csv} ({len(df_sub):,} baris)")
display(df_sub.head(10))
display(df_sub[["p50", "p90"]].describe())

File submission berhasil ditulis: C:\Users\rafai\OneDrive - Telkom University\Documents\UTS ML_REGRESI_KLASIFIKASI\prediksi_regresi.csv (60,000 baris)


,id_paket,p50,p90
0,P83639399,58.7887,73.7796
1,P26553951,103.7298,138.6768
2,P32537117,61.1200,81.0559
3,P28750760,59.7188,78.5229
4,P96580009,49.3828,64.3355
5,P48017366,170.0034,234.4917
6,P43165236,69.0648,87.3494
7,P66505512,49.5898,65.3491
8,P46572172,95.1137,140.4605
9,P88334628,67.2523,83.2617


,p50,p90
count,60000.000000,60000.000000
mean,69.942663,94.710672
std,32.058418,46.169140
min,17.511100,29.810100
25%,47.465725,63.693600
50%,62.757800,82.779650
75%,84.120500,111.341850
max,336.461400,479.465900


## 8. Ringkasan Metodologi, Keterbatasan, & Reproduksibilitas (R3)

- **Anti-Kebocoran & Audit**: Fitur pasca-pickup `jumlah_scan` dibuang penuh; kesalahan satuan `durasi_jam > 1000` diperbaiki (`/ 60.0`) hanya di data latih; sentinel `jarak_km` (`0` / `9999`) dan missingness MNAR `berat_kg` diimputasi ketat di dalam fold latih.
- **Validasi Temporal**: Seluruh tuning hyperparameter dan pemilihan model dilakukan di dalam *expanding-window next-month Time CV* (`train_max < val_min`) dan dibandingkan terhadap Random K-Fold CV.
- **Kategori Kardinalitas Tinggi & Baru**: `kota_tujuan` (405 level) di-encode dengan OOF `TargetEncoder` + frekuensi latih; kurir baru `K13` pada data uji jatuh aman ke prior global dan frekuensi `0.0`.
- **Kalibrasi Ketidakpastian**: Split Conformal Prediction pada fold OOT memastikan cakupan empiris P90 mendekati 90% dengan jaminan `p90 >= p50`.
- **Reproduksi Satu Perintah**: Jalankan `python kode/run_all.py` (atau `python -m kode.regresi` khusus Task 1) dari root proyek menggunakan `.venv`.